# 01 — Spark DataFrames and Schemas

In this notebook, we explore how Apache Spark represents structured data using DataFrames.

We will learn how to create DataFrames, inspect their schemas, and understand how Spark handles rows, columns, and data types.

first run below codes to verify pyspark is installed:

In [3]:
import sys
import pyspark

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)

Python: 3.10.12 (main, Aug 31 2026, 10:18:17) [GCC 11.4.0]
PySpark: 4.3.0


Also below codes to make sure some of pyspark imports actually work:

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count

print("PySpark imports successful!")

PySpark imports successful!


The below snippet creates a Spark session, which is the main entry point for working with Spark DataFrames and other Spark functionality:

In [5]:
spark = (
    SparkSession.builder
    .appName("DataFramesAndSchemas")
    .getOrCreate()
)

print(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 06:39:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Now let's create a small dataset:

In [18]:
employees = [
    (1, "Alice", "Data Engineer", 75000),
    (2, "Bob", "Data Analyst", 65000),
    (3, "Charlie", "Data Engineer", 80000),
    (4, "Diana", "BI Developer", 70000),
    (5, "Eve", "Data Analyst", 68000),
    (6, "Ali", "Data entry", 2000),
    (7, "Nazi", "Data entry", 1000)
]

columns = ["id", "name", "job_title", "salary"]

df = spark.createDataFrame(employees, columns)

df.show()

+---+-------+-------------+------+
| id|   name|    job_title|salary|
+---+-------+-------------+------+
|  1|  Alice|Data Engineer| 75000|
|  2|    Bob| Data Analyst| 65000|
|  3|Charlie|Data Engineer| 80000|
|  4|  Diana| BI Developer| 70000|
|  5|    Eve| Data Analyst| 68000|
|  6|    Ali|   Data entry|  2000|
|  7|   Nazi|   Data entry|  1000|
+---+-------+-------------+------+



In [22]:
filtered_df = df.filter(df.salary > 5000)
filtered_df.show()
filtered_df.printSchema()

+---+-------+-------------+------+
| id|   name|    job_title|salary|
+---+-------+-------------+------+
|  1|  Alice|Data Engineer| 75000|
|  2|    Bob| Data Analyst| 65000|
|  3|Charlie|Data Engineer| 80000|
|  4|  Diana| BI Developer| 70000|
|  5|    Eve| Data Analyst| 68000|
+---+-------+-------------+------+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- salary: long (nullable = true)



In [6]:
import os

data_dir = "/opt/spark/data"

os.listdir(data_dir)

['streaming', 'mllib', 'graphx']

In [6]:
taxi_df = spark.read.parquet(
    "/opt/spark-data/yellow_tripdata_2025-01.parquet"
)
taxi_df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|1       |2025-01-01 00:18:38 |2025-01-01 00:26:59  |1              |1.6          |1         |N                 |229         |237 

In [11]:
taxi_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [17]:
taxi_df.count()

3475226

In [18]:
taxi_df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [20]:
taxi_df.describe().show()

[Stage 21:=======================================>                 (7 + 3) / 10]

+-------+------------------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+---------------------+------------------+--------------------+-------------------+-------------------+
|summary|          VendorID|   passenger_count|    trip_distance|       RatecodeID|store_and_fwd_flag|     PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+---------------------+-------

In [21]:
print("Number of columns:", len(taxi_df.columns))
print("Number of rows:", taxi_df.count())

Number of columns: 20
Number of rows: 3475226


In [16]:
taxi_df.dtypes

[('VendorID', 'int'),
 ('tpep_pickup_datetime', 'timestamp_ntz'),
 ('tpep_dropoff_datetime', 'timestamp_ntz'),
 ('passenger_count', 'bigint'),
 ('trip_distance', 'double'),
 ('RatecodeID', 'bigint'),
 ('store_and_fwd_flag', 'string'),
 ('PULocationID', 'int'),
 ('DOLocationID', 'int'),
 ('payment_type', 'bigint'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('improvement_surcharge', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('Airport_fee', 'double'),
 ('cbd_congestion_fee', 'double')]

In [27]:
taxi_df.createOrReplaceTempView("taxi_df_temp_view")
queried_df=spark.sql("""
        SELECT
            VendorID,
            SUM(passenger_count),
            SUM(trip_distance),
            SUM(fare_amount)
        FROM taxi_df_temp_view
        WHERE trip_distance > 10
        AND passenger_count > 1
        GROUP BY VendorID 
""")
queried_df.show()

+--------+--------------------+------------------+-----------------+
|VendorID|sum(passenger_count)|sum(trip_distance)| sum(fare_amount)|
+--------+--------------------+------------------+-----------------+
|       1|               12110| 80175.20000000003|325880.5800000001|
|       2|              135448| 924226.9599999972|3452381.120000005|
|       7|                  12|             57.17|            255.4|
+--------+--------------------+------------------+-----------------+



In [23]:
selected_df = taxi_df.select(
    taxi_df.VendorID,
    taxi_df.passenger_count,
    taxi_df.trip_distance
)
selected_df.show()

+--------+---------------+-------------+
|VendorID|passenger_count|trip_distance|
+--------+---------------+-------------+
|       1|              1|          1.6|
|       1|              1|          0.5|
|       1|              1|          0.6|
|       2|              3|         0.52|
|       2|              3|         0.66|
|       2|              2|         2.63|
|       1|              0|          0.4|
|       1|              0|          1.6|
|       1|              0|          2.8|
|       2|              1|         1.71|
|       2|              1|         2.29|
|       2|              1|         0.56|
|       2|              3|         1.99|
|       1|              1|          1.1|
|       1|              1|          3.2|
|       1|              3|          2.5|
|       1|              1|          1.9|
|       2|              1|         0.71|
|       2|              1|         0.71|
|       1|              2|          1.2|
+--------+---------------+-------------+
only showing top

In [28]:
from pyspark.sql.functions import when

new_taxi_df = taxi_df.withColumn(
    "high_value_trip",
    when(taxi_df.total_amount > 50, True).otherwise(False)
)

new_taxi_df.select(
    "total_amount",
    "high_value_trip"
).show(10)

+------------+---------------+
|total_amount|high_value_trip|
+------------+---------------+
|        18.0|          false|
|       12.12|          false|
|        12.1|          false|
|         9.7|          false|
|         8.3|          false|
|        24.1|          false|
|       11.75|          false|
|        19.1|          false|
|        27.1|          false|
|        16.4|          false|
+------------+---------------+
only showing top 10 rows


In [29]:
from pyspark.sql.functions import (
    sum,
    avg,
    count,
    min,
    max
)

vendor_stats = (
    taxi_df
    .groupBy("VendorID")
    .agg(
        count("*").alias("total_trips"),
        sum("fare_amount").alias("total_revenue"),
        avg("fare_amount").alias("average_fare"),
        min("fare_amount").alias("minimum_fare"),
        max("fare_amount").alias("maximum_fare")
    )
)

vendor_stats.show()

+--------+-----------+-------------------+------------------+------------+------------+
|VendorID|total_trips|      total_revenue|      average_fare|minimum_fare|maximum_fare|
+--------+-----------+-------------------+------------------+------------+------------+
|       1|     753671|1.390343337999974E7|18.447616241038517|         0.0|   863372.12|
|       7|       1206| 15833.099999999997|13.128606965174127|         3.7|        81.4|
|       2|    2719860|4.544240715000229E7| 16.70762728596409|      -900.0|      2450.9|
|       6|        489| 1451.4499999999941| 2.968200408997943|         0.0|        11.6|
+--------+-----------+-------------------+------------------+------------+------------+



In [33]:
selected_df = taxi_df.filter(taxi_df.fare_amount==863372.12)

selected_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-20 12:07:18|  2025-01-20 12:12:42|              1|          1.6|         1|                 N|         138|    

In [46]:
payment_stats = (
    taxi_df
    .groupBy("VendorID", "payment_type")
    .agg(
        count("*").alias("total_trips"),
        sum("total_amount").alias("total_revenue")
    )
    .orderBy("VendorID","payment_type")
)

payment_stats.show()

+--------+------------+-----------+--------------------+
|VendorID|payment_type|total_trips|       total_revenue|
+--------+------------+-----------+--------------------+
|       1|           0|      88204|  2058700.5200000163|
|       1|           1|     565562|1.5144738050000302E7|
|       1|           2|      86321|  1838243.1199999885|
|       1|           3|       8409|  165022.41000000006|
|       1|           4|       5174|   976363.3099999982|
|       1|           5|          1|                 0.0|
|       2|           0|     451456|   8775408.649999645|
|       2|           1|    1877780| 5.346813506997343E7|
|       2|           2|     303980|   6647512.039999772|
|       2|           3|      15337|  -19287.72000000001|
|       2|           4|      71307|  -91214.51999999995|
|       6|           0|        489|  14985.670000000002|
|       7|           1|       1051|  23741.570000000018|
|       7|           2|        128|  2308.3300000000004|
|       7|           3|        

In [47]:
taxi_df = spark.read.parquet(
    "/opt/spark-data/yellow_tripdata_2025-01.parquet"
)
raw_df = taxi_df

In [49]:
from pyspark.sql.functions import col, count, when

null_counts = raw_df.select(
    *[
        count(
            when(col(c).isNull(), 1)
        ).alias(c)
        for c in raw_df.columns
    ]
)

null_counts.show(truncate=False, vertical=True)

-RECORD 0-----------------------
 VendorID              | 0      
 tpep_pickup_datetime  | 0      
 tpep_dropoff_datetime | 0      
 passenger_count       | 540149 
 trip_distance         | 0      
 RatecodeID            | 540149 
 store_and_fwd_flag    | 540149 
 PULocationID          | 0      
 DOLocationID          | 0      
 payment_type          | 0      
 fare_amount           | 0      
 extra                 | 0      
 mta_tax               | 0      
 tip_amount            | 0      
 tolls_amount          | 0      
 improvement_surcharge | 0      
 total_amount          | 0      
 congestion_surcharge  | 540149 
 Airport_fee           | 540149 
 cbd_congestion_fee    | 0      



In [52]:
raw_df.filter(
    col("passenger_count").isNull()
).show(20, vertical=True)

-RECORD 0------------------------------------
 VendorID              | 2                   
 tpep_pickup_datetime  | 2025-01-01 00:39:17 
 tpep_dropoff_datetime | 2025-01-01 01:10:42 
 passenger_count       | NULL                
 trip_distance         | 6.75                
 RatecodeID            | NULL                
 store_and_fwd_flag    | NULL                
 PULocationID          | 48                  
 DOLocationID          | 42                  
 payment_type          | 0                   
 fare_amount           | 2.32                
 extra                 | 0.0                 
 mta_tax               | 0.5                 
 tip_amount            | 0.0                 
 tolls_amount          | 0.0                 
 improvement_surcharge | 1.0                 
 total_amount          | 14.9                
 congestion_surcharge  | NULL                
 Airport_fee           | NULL                
 cbd_congestion_fee    | 0.0                 
-RECORD 1-------------------------

In [53]:
invalid_records = raw_df.filter(
    (col("trip_distance") <= 0) |
    (col("fare_amount") < 0) |
    (col("passenger_count") < 0)
)

invalid_records.count()

220560

In [55]:
from pyspark.sql.functions import col

missing_df = raw_df.filter(
    col("passenger_count").isNull()
)

print("Missing records:", missing_df.count())

missing_df.show(10, truncate=False, vertical=True)

Missing records: 540149
-RECORD 0------------------------------------
 VendorID              | 2                   
 tpep_pickup_datetime  | 2025-01-01 00:39:17 
 tpep_dropoff_datetime | 2025-01-01 01:10:42 
 passenger_count       | NULL                
 trip_distance         | 6.75                
 RatecodeID            | NULL                
 store_and_fwd_flag    | NULL                
 PULocationID          | 48                  
 DOLocationID          | 42                  
 payment_type          | 0                   
 fare_amount           | 2.32                
 extra                 | 0.0                 
 mta_tax               | 0.5                 
 tip_amount            | 0.0                 
 tolls_amount          | 0.0                 
 improvement_surcharge | 1.0                 
 total_amount          | 14.9                
 congestion_surcharge  | NULL                
 Airport_fee           | NULL                
 cbd_congestion_fee    | 0.0                 
-RECORD 1-

In [56]:
missing_df.groupBy("VendorID").count().show()

+--------+------+
|VendorID| count|
+--------+------+
|       1| 88204|
|       6|   489|
|       2|451456|
+--------+------+



In [57]:
missing_df.groupBy("payment_type").count().show()

+------------+------+
|payment_type| count|
+------------+------+
|           0|540149|
+------------+------+



In [58]:
from pyspark.sql.functions import min, max

missing_df.select(
    min("tpep_pickup_datetime").alias("earliest_pickup"),
    max("tpep_pickup_datetime").alias("latest_pickup")
).show(truncate=False)

+-------------------+-------------------+
|earliest_pickup    |latest_pickup      |
+-------------------+-------------------+
|2025-01-01 00:00:00|2025-01-31 23:59:59|
+-------------------+-------------------+



In [59]:
from pyspark.sql.functions import sum, when

missing_df.select(
    sum(when(col("passenger_count").isNull(), 1).otherwise(0)).alias("passenger_count"),
    sum(when(col("RatecodeID").isNull(), 1).otherwise(0)).alias("RatecodeID"),
    sum(when(col("store_and_fwd_flag").isNull(), 1).otherwise(0)).alias("store_and_fwd_flag"),
    sum(when(col("congestion_surcharge").isNull(), 1).otherwise(0)).alias("congestion_surcharge"),
    sum(when(col("Airport_fee").isNull(), 1).otherwise(0)).alias("Airport_fee")
).show()

+---------------+----------+------------------+--------------------+-----------+
|passenger_count|RatecodeID|store_and_fwd_flag|congestion_surcharge|Airport_fee|
+---------------+----------+------------------+--------------------+-----------+
|         540149|    540149|            540149|              540149|     540149|
+---------------+----------+------------------+--------------------+-----------+



In [60]:
payment_zero_df = raw_df.filter(
    col("payment_type") == 0
)

print("Payment type 0 records:", payment_zero_df.count())

Payment type 0 records: 540149


In [61]:
payment_zero_df.select(
    sum(when(col("passenger_count").isNull(), 1).otherwise(0)).alias("passenger_count_nulls"),
    sum(when(col("RatecodeID").isNull(), 1).otherwise(0)).alias("RatecodeID_nulls"),
    sum(when(col("store_and_fwd_flag").isNull(), 1).otherwise(0)).alias("store_and_fwd_flag_nulls"),
    sum(when(col("congestion_surcharge").isNull(), 1).otherwise(0)).alias("congestion_surcharge_nulls"),
    sum(when(col("Airport_fee").isNull(), 1).otherwise(0)).alias("Airport_fee_nulls")
).show()

+---------------------+----------------+------------------------+--------------------------+-----------------+
|passenger_count_nulls|RatecodeID_nulls|store_and_fwd_flag_nulls|congestion_surcharge_nulls|Airport_fee_nulls|
+---------------------+----------------+------------------------+--------------------------+-----------------+
|               540149|          540149|                  540149|                    540149|           540149|
+---------------------+----------------+------------------------+--------------------------+-----------------+



In [62]:
raw_df.groupBy("payment_type").agg(
    count("*").alias("total_trips"),
    count("passenger_count").alias("passenger_count_present"),
    count("RatecodeID").alias("ratecode_present")
).orderBy("payment_type").show()

+------------+-----------+-----------------------+----------------+
|payment_type|total_trips|passenger_count_present|ratecode_present|
+------------+-----------+-----------------------+----------------+
|           0|     540149|                      0|               0|
|           1|    2444393|                2444393|         2444393|
|           2|     390429|                 390429|          390429|
|           3|      23773|                  23773|           23773|
|           4|      76481|                  76481|           76481|
|           5|          1|                      1|               1|
+------------+-----------+-----------------------+----------------+



In [63]:
from pyspark.sql.functions import col

invalid_condition = (
    (col("payment_type") == 0) |
    (col("trip_distance") <= 0) |
    (col("fare_amount") < 0) |
    (col("passenger_count") < 0) |
    (col("tpep_pickup_datetime").isNull())
)

quarantine_df = raw_df.filter(invalid_condition)

cleaned_df = raw_df.filter(~invalid_condition)

In [64]:
raw_count = raw_df.count()
cleaned_count = cleaned_df.count()
quarantine_count = quarantine_df.count()

print("Raw records:", raw_count)
print("Cleaned records:", cleaned_count)
print("Quarantined records:", quarantine_count)
print("Reconciliation:", cleaned_count + quarantine_count)

Raw records: 3475226
Cleaned records: 2841244
Quarantined records: 633982
Reconciliation: 3475226


In [65]:
quarantine_df.groupBy("payment_type").count().show()

+------------+------+
|payment_type| count|
+------------+------+
|           5|     1|
|           1| 19659|
|           3| 11507|
|           2| 21918|
|           4| 40748|
|           0|540149|
+------------+------+

